# Zimbabwe News Classification


The dataset has just **article** and **category**. Each article includes its original headline, followed by the body. Categories: Politics, Entertainment, Business, Tech, Health and Agriculture. Labels come from publishers and have not been checked by a human. Scores are not verified confidence.

In [3]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

dataset = Path('news_dataset.csv')
categories = ['POLITICS', 'ENTERTAINMENT', 'BUSINESS', 'TECH', 'HEALTH', 'AGRICULTURE']

## 1. Clean the dataset
Keep the headline with its article, remove missing or repeated articles, and save only the two columns we need. Running this cell again will not add the headline twice.

TF-IDF will handle lowercase text, punctuation and English stop words during training, so the saved articles stay readable. Close the CSV in Excel before running this cell.

In [6]:
df = pd.read_csv(dataset)
df.head()

,article,category
0,From spectators to actors: Africa Skills Week ...,TECH
1,Zimbabwean author denies using AI to write awa...,ENTERTAINMENT
2,"Grain in the silos, cash in the bank: Zimbabwe...",AGRICULTURE
3,ZNCC proposes US$0.40 cap on fuel taxes to cut...,BUSINESS
4,Late business tycoon Chivayo at the centre of ...,BUSINESS


In [18]:
df.tail()

,article,category
35377,Banned in Australia\nnd persons associated wit...,BUSINESS
35378,US$10 million paid to IMF\nber and January to ...,BUSINESS
35379,Investment advice for beginners\nrmation that ...,BUSINESS
35380,Fair recovery for stock market\ndex might be c...,BUSINESS
35381,Dead wildlife threatens tourism\ne the animals...,BUSINESS


In [7]:
df = pd.read_csv(dataset)
if 'headline' in df.columns:
    df['article'] = df['headline'].fillna('') + '\n' + df['article'].fillna('')
    
df = df[['article', 'category']].dropna().copy()
df['article'] = df['article'].str.strip()
df['category'] = df['category'].str.strip().str.upper()
df = df[df['article'].ne('') & df['category'].isin(categories)]
df = df.drop_duplicates('article').reset_index(drop=True)
if set(df['category']) != set(categories) or df['category'].value_counts().min() < 5:
    raise ValueError('Each of the six categories needs at least five articles.')
df.to_csv(dataset, index=False)

print(f'{len(df):,} articles ready')
display(df.head())
display(df['category'].value_counts().to_frame('Articles'))

35,382 articles ready


,article,category
0,From spectators to actors: Africa Skills Week ...,TECH
1,Zimbabwean author denies using AI to write awa...,ENTERTAINMENT
2,"Grain in the silos, cash in the bank: Zimbabwe...",AGRICULTURE
3,ZNCC proposes US$0.40 cap on fuel taxes to cut...,BUSINESS
4,Late business tycoon Chivayo at the centre of ...,BUSINESS


,Articles
category,
POLITICS,9677
BUSINESS,9663
HEALTH,7315
ENTERTAINMENT,3679
TECH,2689
AGRICULTURE,2359


## 2. Train the model
Use the first 1,000 words of each article. Keep 20% for testing, and learn the vocabulary from the training data only.

In [8]:
def short_text(text):
    return ' '.join(text.split()[:1000])

X_train, X_test, y_train, y_test = train_test_split(
    df['article'].map(short_text), df['category'],
    test_size=0.2, random_state=42, stratify=df['category'])
model = make_pipeline(
    TfidfVectorizer(stop_words='english', ngram_range=(1, 2),
                    max_features=40000, min_df=2, sublinear_tf=True),
    LogisticRegression(max_iter=500, class_weight='balanced', random_state=42))
model.fit(X_train, y_train)
print(f'Trained on {len(X_train):,} articles; testing on {len(X_test):,}.')

Trained on 28,305 articles; testing on 7,077.


## 3. Check the results
These scores measure agreement with publisher labels. Rewritten stories and publisher writing styles can make a random split look better than performance on new news.

In [9]:
predictions = model.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, predictions):.2%}')
print(classification_report(y_test, predictions, labels=categories, zero_division=0))

Test accuracy: 86.62%
               precision    recall  f1-score   support

     POLITICS       0.92      0.91      0.92      1935
ENTERTAINMENT       0.90      0.94      0.92       736
     BUSINESS       0.85      0.79      0.82      1933
         TECH       0.72      0.84      0.78       538
       HEALTH       0.92      0.90      0.91      1463
  AGRICULTURE       0.71      0.80      0.75       472

     accuracy                           0.87      7077
    macro avg       0.84      0.86      0.85      7077
 weighted avg       0.87      0.87      0.87      7077



## 4. Predict a category
Enter a headline, an article, or both. The model always chooses one of the six categories.

In [10]:
def classify_article(headline='', article=''):
    if not isinstance(headline, str) or not isinstance(article, str):
        raise ValueError('The headline and article must be text.')
    text = short_text(headline + ' ' + article)
    if not model[0].transform([text]).nnz:
        raise ValueError('Please enter an English headline or article with recognised words.')
    scores = pd.Series(model.predict_proba([text])[0], index=model.classes_)
    scores = scores.sort_values(ascending=False)
    return scores.index[0], scores

## 5. Paste your headline and article here
Replace the text inside the triple quotes. After Run All, rerun only this last cell for each new article.

In [21]:
my_headline = """TelOne launches 4G feature phone to rival Econet’s Smart Kambudzi
"""

my_article = """ TelOne has joined in on the feature phone game by announcing that it will be offering 
an MLS Easy TS 4G Smart feature phone that will go head to head with Econet’s Smart 4G Kambudzi.
"""

try:
    category, scores = classify_article(my_headline, my_article)
    print('Category:', category)
    display(scores.rename('Score').to_frame())
except ValueError as error:
    print(error)


Category: TECH


,Score
TECH,0.515078
BUSINESS,0.187865
ENTERTAINMENT,0.118424
POLITICS,0.067024
AGRICULTURE,0.061724
HEALTH,0.049887
